In [1]:
import kagglehub
import pandas as pd
import os
import ast
import html
import re
import string
import unicodedata
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import nltk 
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/rpesic/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/rpesic/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

Loading raw data

In [4]:
path = kagglehub.dataset_download("shuyangli94/foodcom-recipes-with-search-terms-and-tags")
csv_path = os.path.join(path, 'recipes_w_search_terms.csv')
df = pd.read_csv(csv_path)

def get_cuisine(x):
    if not isinstance(x, str):
        return None
    x_lower = x.lower()
    if 'italian' in x_lower:
        return 'Italian'
    if 'indian' in x_lower:
        return 'Indian'
    return None

italian_indian_df = df[df['search_terms'].apply(lambda x: isinstance(x, str) and ('italian' in x.lower() or 'indian' in x.lower()))].copy()
italian_indian_df['cuisine'] = italian_indian_df['search_terms'].apply(get_cuisine)
final_df = italian_indian_df[['name', 'steps', 'cuisine']].reset_index(drop=True)
final_df['steps'] = final_df['steps'].apply(lambda x: ' '.join(ast.literal_eval(x)) if isinstance(x, str) else x)
final_df['name'] = final_df['name'].str.lower()
final_df['steps'] = final_df['steps'].str.lower()

indian_all = final_df[final_df['cuisine'] == 'Indian']
italian_sample = final_df[final_df['cuisine'] == 'Italian'].sample(n=len(indian_all), random_state=42)

final_df = pd.concat([italian_sample, indian_all], ignore_index=True)
final_df.to_csv('./data/recipes_raw.csv')

### Class imbalance handling comparison

Before downstream processing, `italian_indian_df` above still holds the raw, imbalanced counts (Italian recipes outnumber Indian ~2.6:1 in this dataset). Rather than assuming undersampling is the right call, this compares it against the other standard strategies (no balancing, class-weighting, oversampling, SMOTE) on a quick BoW + logistic regression cuisine classifier, scored by ROC-AUC on a held-out test set that keeps the *original* imbalanced distribution (so the score reflects a realistic, non-resampled evaluation regardless of which strategy trained the model).

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler, SMOTE

imbalance_df = italian_indian_df.dropna(subset=['cuisine'])
y_imb_all = (imbalance_df['cuisine'] == 'Indian').astype(int)  # minority class = 1

X_text_train, X_text_test, y_train_imb, y_test_imb = train_test_split(
    imbalance_df['steps'], y_imb_all, test_size=0.2, stratify=y_imb_all, random_state=42
)

imb_vectorizer = CountVectorizer(max_features=5000, stop_words='english')
X_train_imb = imb_vectorizer.fit_transform(X_text_train)
X_test_imb = imb_vectorizer.transform(X_text_test)

def imbalance_strategy_auc(X_res, y_res, class_weight=None):
    clf = LogisticRegression(max_iter=1000, class_weight=class_weight)
    clf.fit(X_res, y_res)
    proba = clf.predict_proba(X_test_imb)[:, 1]
    return roc_auc_score(y_test_imb, proba)

X_under, y_under = RandomUnderSampler(random_state=42).fit_resample(X_train_imb, y_train_imb)
X_over, y_over = RandomOverSampler(random_state=42).fit_resample(X_train_imb, y_train_imb)
X_smote, y_smote = SMOTE(random_state=42).fit_resample(X_train_imb, y_train_imb)

imbalance_auc_scores = {
    'No balancing': imbalance_strategy_auc(X_train_imb, y_train_imb),
    'class_weight=balanced': imbalance_strategy_auc(X_train_imb, y_train_imb, class_weight='balanced'),
    'Random undersampling': imbalance_strategy_auc(X_under, y_under),
    'Random oversampling': imbalance_strategy_auc(X_over, y_over),
    'SMOTE': imbalance_strategy_auc(X_smote, y_smote),
}

for name, score in imbalance_auc_scores.items():
    print(f'{name}: AUC = {score:.4f}')

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.bar(imbalance_auc_scores.keys(), imbalance_auc_scores.values(), color='#4C72B0')
plt.ylabel('ROC-AUC')
plt.title('Cuisine classification AUC by class-imbalance handling strategy')
plt.xticks(rotation=20, ha='right')
plt.ylim(0.9, 1.0)
plt.tight_layout()
plt.show()

**Conclusion:** all five strategies score within ~0.005 AUC of each other (0.983-0.988) - cuisine is easy to tell apart from word choice alone (e.g. curry/masala vs. parmesan/basil), so this task isn't actually sensitive to how the imbalance is handled. Undersampling (the strategy used above) performs on par with the more complex alternatives (oversampling, SMOTE, class-weighting) while also being the simplest and cheapest at training time for the downstream seq2seq model - so it's kept as-is.

In [5]:
non_food_signals = ['scalp', 'shampoo', 'massage your']
final_df['suspicious'] = final_df['steps'].str.lower().apply(
    lambda x: any(w in x for w in non_food_signals))
print(final_df[final_df['suspicious']][['name', 'cuisine']])

final_df = final_df[~final_df['suspicious']].drop(columns='suspicious').reset_index(drop=True)

                                                    name cuisine
10911  magical transformation from very rough and dry...  Indian
12027             homemade scrub to get rid of dead skin  Indian
13107                   silky hair with an egg treatment  Indian


Basic checks

In [6]:
final_df["name"] = (
    final_df["name"]
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)
final_df["steps"] = (
    final_df["steps"]
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

print('Number of null rows\n', final_df.isna().sum())
print('Number of duplicated rows', final_df.duplicated(subset='steps').sum())
df_no_dub = final_df.drop_duplicates(subset='steps').reset_index(drop=True)

Number of null rows
 name       0
steps      0
cuisine    0
dtype: int64
Number of duplicated rows 14


Text cleaning (HTML entities, invisible/control characters)

In [7]:
# zero-width space (200b), line/paragraph separator (2028/2029), BOM (feff), nbsp (a0)
_invisible = [0x200b, 0x2028, 0x2029, 0xfeff, 0xa0]
INVISIBLE_CHARS = re.compile('[' + ''.join(chr(c) for c in _invisible) + ']')
CONTROL_CHARS = re.compile('[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]')

def clean_text(text):
    if not isinstance(text, str):
        return text
    text = html.unescape(text)
    text = unicodedata.normalize('NFKC', text)
    text = INVISIBLE_CHARS.sub(' ', text)
    text = CONTROL_CHARS.sub(' ', text)
    text = re.sub(r'-{2,}', '-', text)  # collapse repeated hyphens (e.g. "----") into one
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_clean = df_no_dub.copy()
df_clean['name'] = df_clean['name'].apply(clean_text)
df_clean['steps'] = df_clean['steps'].apply(clean_text)

In [8]:
from nltk.tokenize import sent_tokenize

noise_signals = ['read more', 'hit play', 'www']

def remove_noisy_sentences(text):
    sentences = sent_tokenize(text)
    kept = [s for s in sentences if not any(w in s.lower() for w in noise_signals)]
    return ' '.join(kept)

df_clean['steps'] = df_clean['steps'].apply(remove_noisy_sentences)

Tokenization

In [9]:
df_clean['steps_tokens'] = df_clean['steps'].apply(word_tokenize)
df_clean.to_csv('./data/recipes_tokenized.csv')

In [10]:
stop_words = set(stopwords.words('english'))

def remove_stopwords(tokens):
    return [t for t in tokens if t not in stop_words]

def remove_punctuation(tokens):
    return [token for token in tokens if token not in string.punctuation]

# steps_tokens keeps stopwords here - it's the seq2seq target; stopwords are removed
# later from a copy (steps_tokens_bow) for the BoW classifier. Punctuation carries no
# useful signal either way, so it's dropped from steps_tokens too.
df_clean['steps_tokens'] = df_clean['steps_tokens'].apply(remove_punctuation)

Statistical analysis

In [11]:
df_clean['number_of_tokens'] = df_clean['steps_tokens'].apply(len)
print(df_clean['number_of_tokens'].describe())

max_len = int(df_clean['number_of_tokens'].quantile(0.95))
df_clean_no_outliers = df_clean[
    df_clean['number_of_tokens'].between(13, max_len - 1)
].copy()
print(df_clean_no_outliers['cuisine'].value_counts())

count    13095.000000
mean       123.042688
std         84.319923
min          1.000000
25%         68.000000
50%        104.000000
75%        155.000000
max       1220.000000
Name: number_of_tokens, dtype: float64
cuisine
Indian     6221
Italian    6118
Name: count, dtype: int64


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df_clean_no_outliers['cuisine'].value_counts().plot(kind='bar', ax=axes[0], color=['#4C72B0', '#DD8452'])
axes[0].set_title('Class balance (after outlier filtering)')
axes[0].set_xlabel('cuisine')
axes[0].set_ylabel('count')
axes[0].tick_params(axis='x', rotation=0)

df_clean['number_of_tokens'].hist(bins=50, ax=axes[1])
axes[1].axvline(13, color='red', linestyle='--', label='cutoff kept')
axes[1].axvline(max_len, color='red', linestyle='--')
axes[1].set_title('Recipe length distribution (tokens), before outlier filtering')
axes[1].set_xlabel('number_of_tokens')
axes[1].legend()

plt.tight_layout()
plt.show()

Additional steps_tokens cleaning (numbers, units, artifacts, fractions, hyphens)

In [12]:
MEASUREMENT_WORDS = {
    "cup", "cups",
    "tbsp", "tablespoon", "tablespoons",
    "tsp", "teaspoon", "teaspoons",
    "oz", "ounce", "ounces",
    "lb", "lbs", "pound", "pounds",
    "qt", "quart", "quarts",
    "pint", "pints",
    "g", "kg", "mg",
    "ml", "l",
    "inch", "inches",
    "degree", "degrees",
    "minute", "minutes",
    "hour", "hours"
}

def remove_numbers_measurements(tokens):
    cleaned = []
    for token in tokens:
        token = token.lower()
        # remove pure numbers and fractions
        if re.fullmatch(r"[\d¼½¾⁄/.-]+", token):
            continue
        if token in MEASUREMENT_WORDS:
            continue
        cleaned.append(token)
    return cleaned

ARTIFACTS = {
    "'s", "'re", "'ve", "'ll", "'d", "'m", "n't", "--"
}

def remove_artifacts(tokens):
    return [t for t in tokens if t not in ARTIFACTS]

def normalize_unicode_fractions(tokens):
    replacements = {"½": "1/2", "¼": "1/4", "¾": "3/4", "⁄": "/"}
    normalized = []
    for token in tokens:
        for old, new in replacements.items():
            token = token.replace(old, new)
        normalized.append(token)
    return normalized

def split_hyphenated(tokens):
    output = []
    for token in tokens:
        output.extend(token.replace("-", " ").split())
    return output

# normalize_unicode_fractions is lossless, so it's also applied to steps_tokens (the seq2seq target).
# The other transforms (dropping quantities/measurements, artifacts, splitting hyphenated words)
# change the actual content/surface form, so they stay BoW-only in steps_tokens_bow.
df_clean_no_outliers["steps_tokens"] = df_clean_no_outliers["steps_tokens"].apply(normalize_unicode_fractions)

df_clean_no_outliers["steps_tokens_bow"] = df_clean_no_outliers["steps_tokens"].apply(remove_numbers_measurements)
df_clean_no_outliers["steps_tokens_bow"] = df_clean_no_outliers["steps_tokens_bow"].apply(remove_artifacts)
df_clean_no_outliers["steps_tokens_bow"] = df_clean_no_outliers["steps_tokens_bow"].apply(split_hyphenated)

Branch: BoW features for the content classifier (steps_tokens_bow)

steps_tokens stays untouched as the seq2seq target. steps_tokens_bow is a copy with stopwords/punctuation removed, used only for the BoW/multi-task classifier.

In [13]:
df_clean_no_outliers["steps_tokens_bow"] = df_clean_no_outliers["steps_tokens_bow"].apply(remove_stopwords)
df_clean_no_outliers["steps_tokens_bow"] = df_clean_no_outliers["steps_tokens_bow"].apply(remove_punctuation)

df_clean_no_outliers.to_csv('./data/recipes_final.csv')

Train/val/test split

In [14]:
from sklearn import model_selection

df = pd.read_csv('./data/recipes_final.csv', index_col=0)
df['steps_tokens'] = df['steps_tokens'].apply(ast.literal_eval)
df['steps_tokens_bow'] = df['steps_tokens_bow'].apply(ast.literal_eval)

X = df[['steps_tokens', 'steps_tokens_bow']]
y = df['cuisine']

X_train, X_temp, y_train, y_temp = model_selection.train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)
X_val, X_test, y_val, y_test = model_selection.train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

In [15]:
train_df = X_train.join(y_train)
val_df = X_val.join(y_val)
test_df = X_test.join(y_test)

train_df.to_csv('./data/recipes_train.csv')
val_df.to_csv('./data/recipes_val.csv')
test_df.to_csv('./data/recipes_test.csv')

print(train_df.shape, val_df.shape, test_df.shape)

(8637, 3) (1851, 3) (1851, 3)


Word2Vec

In [16]:
from gensim.models import Word2Vec

WORD2VEC_PATH = './data/models/word2vec.wordvectors'

# keep existing vectors on disk if present - retraining would give the seq2seq model
# new random word vectors, making any saved checkpoint (trained on the old vectors)
# incompatible even though its weight shapes still match
if not os.path.exists(WORD2VEC_PATH):
    w2v_model = Word2Vec(
        sentences=train_df['steps_tokens'],
        vector_size=150,
        window=5,
        min_count=2,
        workers=4,
        epochs=15
    )
    w2v_model.wv.save(WORD2VEC_PATH)
else:
    print(f'{WORD2VEC_PATH} already exists, skipping retraining')

./data/models/word2vec.wordvectors already exists, skipping retraining


BoW

In [17]:
from sklearn.feature_extraction.text import CountVectorizer
import joblib

def identity_analyzer(tokens):
    return tokens

BOW_VECTORIZER_PATH = './data/models/bow_vectorizer.joblib'

# keep the existing vectorizer if present - refitting can shift the BoW vocabulary
# indices, which would make the saved checkpoint's content_classifier (trained against
# the old BoW vocabulary) inconsistent
if os.path.exists(BOW_VECTORIZER_PATH):
    vectorizer = joblib.load(BOW_VECTORIZER_PATH)
    X_train_bow = vectorizer.transform(train_df['steps_tokens_bow'])
    print(f'{BOW_VECTORIZER_PATH} already exists, reusing it')
else:
    vectorizer = CountVectorizer(analyzer=identity_analyzer)
    X_train_bow = vectorizer.fit_transform(train_df['steps_tokens_bow'])
    joblib.dump(vectorizer, BOW_VECTORIZER_PATH)

./data/models/bow_vectorizer.joblib already exists, reusing it


## Output

This notebook produces everything the model needs, saved under `data/` (gitignored - rerun this notebook to regenerate it locally):
- `data/recipes_train.csv`, `data/recipes_val.csv`, `data/recipes_test.csv`
- `data/models/word2vec.wordvectors`
- `data/models/bow_vectorizer.joblib`

Word2Vec/BoW artifacts are only (re)built if missing - retraining them would silently invalidate any saved model checkpoint trained against the old vectors/vocabulary.

Continue in **`02_model.ipynb`**.